In [51]:
import torch

In [52]:
# Load Pride and Prejudice from Project Gutenberg and convert to word-based token
import urllib.request
import re

url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
response = urllib.request.urlopen(url)
raw_text = response.read().decode('utf-8-sig')

# Find where the actual book starts (after Gutenberg header)
# Look for "It is a truth universally acknowledged" - the famous opening line
start = raw_text.find("It is a truth universally acknowledged")
if start == -1:
    start = 0

# Find where the book ends
end = raw_text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
if end == -1:
    end = len(raw_text)

text = raw_text[start:end].strip()

# Remove illustration markers and extra whitespace
text = re.sub(r'\[Illustration[^\]]*\]', '', text)
text = re.sub(r'\s+', ' ', text)
# Remove publisher info at end
text = text[0:-80]


In [53]:
# Model Hyperparams
vocab = sorted(list(set(text))) + ["<UNK>", "<PAD>"]
d_model = 256
seq_len = 256
n_heads = 4
n_layers = 4


In [54]:
# Mappings
itos = {i: vocab[i] for i in range(len(vocab))}  # index to string
stoi = {vocab[i]: i for i in range(len(vocab))}  # string to index

def create_causal_mask(seq_len, device):
    """Lower triangular mask for GPT self-attention"""

    # Hides output ahead of the current place in sequence so it cannot cheat
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // self.n_heads
        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.O = torch.nn.Linear(d_model, d_model)
        self.dropout = torch.nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """Performs The forward pass of the Multi-Head Attention
           Takes in an embedding x and returns attention vals
        """
        batch_size, seq_len, _ = x.shape

        # Linear transformation to Query, Key, Value vals
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into heads and realign for efficient parallel calc
        Q = Q.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)

        context = self.attention(Q, K, V, mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.O(context)

    def attention(self, Q, K, V, mask=None):
        """Performs the full attention calculation for all n_heads heads """
        attention_vals = torch.matmul(Q, K.transpose(-2, -1))
        attention_vals = attention_vals / torch.sqrt(self.dk ** 0.5)


        if mask is not None:
            # Applies any of our 3 masks
            attention_vals = attention_vals.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = torch.nn.functional.softmax(attention_vals, dim=-1)
        
        # Applies drop out for regularization then returns attention weights
        attention_weights = self.dropout(attention_weights)

        return torch.matmul(attention_weights, V)

In [56]:
class FeedForward(torch.nn.Module):

    def __init__(self, d_model, d_ff, dropout):
         super().__init__()

         self.network = torch.nn.Sequential(
              torch.nn.Linear(d_model, d_ff),
              torch.nn.GELU(),
              torch.nn.Dropout(dropout),
              torch.nn.Linear(d_ff, d_model),

         )
    
    def forward(self, x):
        return self.network(x)

In [57]:
class TransformerBlock(torch.nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.dropout = torch.nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn.forward(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


In [ ]:
class GPT(torch.nn.Module):
    def __init__(self, d_model, vocab, seq_len, n_heads, n_layers, d_ff, dropout):
        super().__init__()

        self.seq_len = seq_len
    
        # Embeddings
        self.token_embedding_layer = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=d_model)
        self.pos_embedding_layer = torch.nn.Embedding(num_embeddings=seq_len, embedding_dim=d_model)

        self.dropout = torch.nn.Dropout(dropout)
        
        # Transformer Blocks
        self.blocks = torch.nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

        self.final_norm = torch.nn.LayerNorm(d_model)
        self.head = torch.nn.Linear(d_model, len(vocab), bias=False)
    
    def forward_pass(self, x, mask=None):
